> **Historical record (ADR-20).** This notebook documents the (superseded) urban go/no-go gate and candidate comparison. The active event is the Brahmaputra riverine flood of 23 June 2023 (see notebooks 02-05 and `code/config/config.yaml`).

# 01 — Event signal gate and candidate comparison

Cell-by-cell view of:

- `code/preprocessing/day3_gate.py`
- `code/preprocessing/candidate_events.py`
- `code/preprocessing/candidate_s2.py`

These are **diagnostics, not final results.** Permanent Brahmaputra water is dark
in every SAR scene, so every test excludes JRC permanent water.

> The April 2026 SAR/S2 subsets were archived to `data/candidates/april2026/`
> after the pivot. `data/raw/` now holds the 31 May 2025 event.

In [1]:
# Setup — run this cell first.
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "code" / "config" / "config.yaml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "code" / "preprocessing"))
print("project root:", ROOT)

project root: D:\DIVP PROJECT


## Step 1 — Reproduce the 20 Apr 2026 negative result

Set `EVENT_DIR` to the archived April subsets. The static layers (JRC, land cover,
DEM) stay in `data/raw/`. Outputs are written with the `negative_april2026` label
so the active 31 May gate table is not overwritten.

In [2]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform_bounds
from scipy import ndimage as ndi
from skimage.filters import threshold_otsu

from common import load_config, load_json, rel_path

NODATA = -32768.0
DARK_DB = -16.0
CHANGE_DB = -2.0

EVENT_DIR = ROOT / "data" / "candidates" / "april2026"
RUN_LABEL = "negative_april2026"
print("event dir:", EVENT_DIR)


def to_db(arr):
    out = np.full(arr.shape, np.nan, dtype="float32")
    valid = (arr > 0) & (arr != NODATA)
    out[valid] = 10.0 * np.log10(arr[valid])
    return out


def crop_bounds(src_transform, height, width, bbox_wgs84, dst_crs):
    minx, miny, maxx, maxy = transform_bounds(
        "EPSG:4326", dst_crs, *bbox_wgs84, densify_pts=21
    )
    inv = ~src_transform
    col_min, row_max = inv * (minx, miny)
    col_max, row_min = inv * (maxx, maxy)
    c0 = max(0, int(np.floor(min(col_min, col_max))))
    c1 = min(width, int(np.ceil(max(col_min, col_max))))
    r0 = max(0, int(np.floor(min(row_min, row_max))))
    r1 = min(height, int(np.ceil(max(row_min, row_max))))
    return slice(r0, r1), slice(c0, c1)


def read(path):
    with rasterio.open(path) as src:
        return src.read(1), src.transform, src.crs


def clean_mask(mask, min_px):
    opened = ndi.binary_opening(mask, structure=np.ones((3, 3), dtype=bool))
    labels, count = ndi.label(opened)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_px
    keep[0] = False
    return keep[labels]

event dir: D:\DIVP PROJECT\data\candidates\april2026


In [3]:
cfg = load_config()
root = Path(cfg["_root"])
raw = rel_path(cfg, "data_raw")

pre_lin, transform, crs = read(EVENT_DIR / "sentinel1" / "s1_rtc_pre_vv.tif")
post_lin, _, _ = read(EVENT_DIR / "sentinel1" / "s1_rtc_post_vv.tif")
b03, _, _ = read(EVENT_DIR / "sentinel2" / "s2_post_B03.tif")
b11, _, _ = read(EVENT_DIR / "sentinel2" / "s2_post_B11.tif")
scl, _, _ = read(EVENT_DIR / "sentinel2" / "s2_post_SCL.tif")
jrc, _, _ = read(raw / "reference" / "jrc_occurrence_aoi.tif")
lc, _, _ = read(raw / "landcover" / "worldcover_aoi.tif")
dem, _, _ = read(raw / "dem" / "dem_aoi.tif")

pre_db = to_db(pre_lin)
post_db = to_db(post_lin)
diff = post_db - pre_db
valid = np.isfinite(pre_db) & np.isfinite(post_db)

dem_f = dem.astype("float32")
dy, dx = np.gradient(dem_f, 10.0, 10.0)
slope = np.degrees(np.arctan(np.hypot(dx, dy)))
slope[dem_f <= -9000] = np.nan
slope_ok = np.isfinite(slope) & (slope <= cfg["masks"]["slope_deg_max"])

permanent = jrc.astype("float32") >= cfg["masks"]["permanent_water"]["occurrence_threshold"]
urban = lc == 50
nonperm = valid & ~permanent

clear = np.isin(scl, [2, 4, 5, 6, 7]) & (b03 > 0) & (b11 > 0)
denom = b03.astype("float32") + b11.astype("float32")
mndwi = np.full(b03.shape, np.nan, dtype="float32")
ok = clear & (denom > 0)
mndwi[ok] = (b03[ok].astype("float32") - b11[ok].astype("float32")) / denom[ok]
s2_water = np.isfinite(mndwi) & (mndwi > cfg["validation"]["mndwi"]["threshold"])
s2_water_np = s2_water & ~permanent
s2_clear_np = clear & ~permanent
print("loaded inputs; valid fraction:", round(float(valid.mean()), 4))

loaded inputs; valid fraction: 1.0


In [4]:
otsu = float(threshold_otsu(post_db[valid]))
post_dark_otsu = valid & (post_db < otsu)
post_dark16 = valid & (post_db < DARK_DB)
change_neg = valid & (diff < CHANGE_DB)
floodlike = post_dark16 & change_neg & ~permanent & slope_ok & valid
floodlike_clean = clean_mask(
    floodlike, cfg["flood_masks"]["morphology"]["min_component_px"]
)


def frac(mask, base):
    total = int(base.sum())
    return float(mask[base].mean()) if total else float("nan")


px_km2 = (cfg["grid"]["resolution_m"] ** 2) / 1e6
stats = {
    "otsu_threshold_db": round(otsu, 3),
    "valid_fraction": round(float(valid.mean()), 4),
    "s2_clear_fraction": round(float(clear.mean()), 4),
    "s2_water_fraction_clear": round(frac(s2_water, clear), 4),
    "s2_clear_nonpermanent_fraction": round(float(s2_clear_np.mean()), 4),
    "s2_water_fraction_clear_nonpermanent": round(frac(s2_water_np, s2_clear_np), 4),
    "post_dark16_fraction_nonpermanent": round(frac(post_dark16, nonperm), 4),
    "pre_dark16_fraction_nonpermanent": round(frac(valid & (pre_db < DARK_DB), nonperm), 4),
    "change_lt_minus2_fraction_nonpermanent": round(frac(change_neg, nonperm), 4),
    "post_dark16_fraction_urban_nonpermanent": round(frac(post_dark16, nonperm & urban), 4),
    "change_lt_minus2_fraction_urban_nonpermanent": round(frac(change_neg, nonperm & urban), 4),
    "floodlike_fraction_nonpermanent": round(frac(floodlike, nonperm), 4),
    "floodlike_area_km2": round(float(floodlike.sum()) * px_km2, 3),
    "floodlike_clean_area_km2": round(float(floodlike_clean.sum()) * px_km2, 3),
    "post_dark16_where_s2_water_np": round(frac(post_dark16, s2_water_np), 4),
    "post_dark16_where_s2_nonwater_np": round(frac(post_dark16, s2_clear_np & ~s2_water_np), 4),
    "change_neg_where_s2_water_np": round(frac(change_neg, s2_water_np), 4),
    "change_neg_where_s2_nonwater_np": round(frac(change_neg, s2_clear_np & ~s2_water_np), 4),
}
for key, value in stats.items():
    print(f"  {key}: {value}")

  otsu_threshold_db: -11.204
  valid_fraction: 1.0
  s2_clear_fraction: 0.9141
  s2_water_fraction_clear: 0.1267
  s2_clear_nonpermanent_fraction: 0.8216
  s2_water_fraction_clear_nonpermanent: 0.0551
  post_dark16_fraction_nonpermanent: 0.0198
  pre_dark16_fraction_nonpermanent: 0.0187
  change_lt_minus2_fraction_nonpermanent: 0.1553
  post_dark16_fraction_urban_nonpermanent: 0.0002
  change_lt_minus2_fraction_urban_nonpermanent: 0.1269
  floodlike_fraction_nonpermanent: 0.0058
  floodlike_area_km2: 1.224
  floodlike_clean_area_km2: 0.066
  post_dark16_where_s2_water_np: 0.3609
  post_dark16_where_s2_nonwater_np: 0.0016
  change_neg_where_s2_water_np: 0.2411
  change_neg_where_s2_nonwater_np: 0.1492


In [5]:
bbox = [91.70, 26.13, 91.80, 26.19]
rs, cs = crop_bounds(transform, pre_lin.shape[0], pre_lin.shape[1], bbox, crs)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
panels = [
    (axes[0, 0], pre_db[rs, cs], "Pre-event VV (dB)", "gray", -22, 0),
    (axes[0, 1], post_db[rs, cs], "Post-event VV (dB)", "gray", -22, 0),
    (axes[0, 2], diff[rs, cs], "Post - pre (dB)", "RdBu", -8, 8),
    (axes[0, 3], mndwi[rs, cs], "S2 MNDWI (clear)", "RdYlBu", -0.5, 0.5),
    (axes[1, 0], np.where(post_dark16[rs, cs], 1, 0), "Post VV < -16 dB", "gray", 0, 1),
    (axes[1, 1], np.where(change_neg[rs, cs], 1, 0), "Change < -2 dB", "gray", 0, 1),
    (axes[1, 2], np.where(floodlike_clean[rs, cs], 1, 0), "Flood-like (cleaned)", "gray", 0, 1),
    (axes[1, 3], np.where(s2_water_np[rs, cs], 1, 0), "S2 water (non-permanent)", "gray", 0, 1),
]
for ax, data, title, cmap, vmin, vmax in panels:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("Event gate — Bharalu / central Guwahati (diagnostic only)")
fig.tight_layout()

fig_dir = rel_path(cfg, "outputs") / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_dir / f"{RUN_LABEL}_gate.png", dpi=150)
plt.close(fig)

table = rel_path(cfg, "outputs") / "tables" / f"{RUN_LABEL}_gate.csv"
table.parent.mkdir(parents=True, exist_ok=True)
with open(table, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["metric", "value"])
    for key, value in stats.items():
        writer.writerow([key, value])

separation = stats["post_dark16_where_s2_water_np"] - stats["post_dark16_where_s2_nonwater_np"]
coherent_area = stats["floodlike_clean_area_km2"]
print(f"non-permanent SAR-vs-S2 separation: {separation:.4f}")
print(f"coherent flood-like area: {coherent_area:.3f} km^2")
print("wrote", table)
print("wrote", fig_dir / f"{RUN_LABEL}_gate.png")

non-permanent SAR-vs-S2 separation: 0.3593
coherent flood-like area: 0.066 km^2
wrote D:\DIVP PROJECT\outputs\tables\negative_april2026_gate.csv
wrote D:\DIVP PROJECT\outputs\figures\negative_april2026_gate.png


## Step 2 — Fetch and gate the two fallback candidates

Candidate A: 19–20 May 2025 (RTC 7 May / 19 May, ASC rel 41).
Candidate B: 31 May 2025 (RTC 21 May / 2 June, DESC rel 77).

In [6]:
from datetime import datetime, timedelta

from rasterio.enums import Resampling

from common import (
    combine_mosaics, get_item, read_reprojected, sign_item, stac_client,
    target_grid, write_geotiff,
)

CANDIDATES = {
    "may19": {
        "label": "19-20 May 2025",
        "pre_date": "2025-05-07",
        "post_date": "2025-05-19",
        "orbit_state": "ascending",
        "rel_orbit": 41,
    },
    "may31": {
        "label": "31 May 2025",
        "pre_date": "2025-05-21",
        "post_date": "2025-06-02",
        "orbit_state": "descending",
        "rel_orbit": 77,
    },
}


def next_day(date):
    d = datetime.strptime(date, "%Y-%m-%d") + timedelta(days=1)
    return d.strftime("%Y-%m-%d")


def find_s1(client, cfg, date, orbit_state, rel_orbit):
    items = list(
        client.search(
            collections=[cfg["stac"]["collections"]["s1_rtc"]],
            bbox=cfg["aoi"]["bbox"],
            datetime=f"{date}/{next_day(date)}",
            max_items=20,
        ).items()
    )
    for item in items:
        if (
            item.properties.get("sat:orbit_state") == orbit_state
            and item.properties.get("sat:relative_orbit") == rel_orbit
        ):
            return item
    raise RuntimeError(f"No S1 RTC for {date} {orbit_state} rel{rel_orbit}")


def clean(mask, min_px):
    opened = ndi.binary_opening(mask, structure=np.ones((3, 3), dtype=bool))
    labels, _ = ndi.label(opened)
    sizes = np.bincount(labels.ravel())
    keep = sizes >= min_px
    keep[0] = False
    return keep[labels]


def fetch_event(cfg, client, name, spec, dst_crs, dst_transform, width, height, root):
    out_dir = Path(cfg["_root"]) / "data" / "candidates" / name
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== {name}: {spec['label']} ===")
    for role, date in (("pre", spec["pre_date"]), ("post", spec["post_date"])):
        item = sign_item(find_s1(client, cfg, date, spec["orbit_state"], spec["rel_orbit"]))
        print(f"  {role}: {item.id}")
        for pol in ("vv", "vh"):
            arr = read_reprojected(
                item.assets[pol].href, dst_crs, dst_transform, width, height,
                Resampling.bilinear, dst_nodata=-32768.0,
            )
            if arr is None:
                raise RuntimeError(f"No overlap {name} {role} {pol}")
            path = out_dir / (f"pre_{pol}.tif" if role == "pre" else f"post_{pol}.tif")
            write_geotiff(path, arr, dst_transform, dst_crs, -32768.0)
            print(f"    wrote {path.name}")
    return out_dir


def gate(cfg, name, spec, out_dir, jrc, lc, dem, px_km2, min_px):
    pre = to_db(rasterio.open(out_dir / "pre_vv.tif").read(1).astype("float32"))
    post = to_db(rasterio.open(out_dir / "post_vv.tif").read(1).astype("float32"))
    diff = post - pre
    valid = np.isfinite(pre) & np.isfinite(post)

    dem_f = dem.astype("float32")
    dy, dx = np.gradient(dem_f, 10.0, 10.0)
    slope = np.degrees(np.arctan(np.hypot(dx, dy)))
    slope[dem_f <= -9000] = np.nan
    slope_ok = np.isfinite(slope) & (slope <= cfg["masks"]["slope_deg_max"])
    permanent = jrc.astype("float32") >= cfg["masks"]["permanent_water"]["occurrence_threshold"]
    urban = lc == 50
    nonperm = valid & ~permanent

    dark16 = valid & (post < -16.0)
    change_neg = valid & (diff < -2.0)
    floodlike = dark16 & change_neg & ~permanent & slope_ok
    floodlike_clean = clean(floodlike, min_px)
    floodlike_urban = floodlike_clean & urban

    def frac(mask, base):
        return float(mask[base].mean()) if base.sum() else float("nan")

    stats = {
        "event": name, "label": spec["label"],
        "pre_date": spec["pre_date"], "post_date": spec["post_date"],
        "post_dark16_nonpermanent": round(frac(dark16, nonperm), 4),
        "post_dark16_urban_nonpermanent": round(frac(dark16, nonperm & urban), 4),
        "change_lt_minus2_nonpermanent": round(frac(change_neg, nonperm), 4),
        "change_lt_minus2_urban_nonpermanent": round(frac(change_neg, nonperm & urban), 4),
        "floodlike_area_km2": round(float(floodlike.sum()) * px_km2, 3),
        "floodlike_clean_area_km2": round(float(floodlike_clean.sum()) * px_km2, 3),
        "floodlike_clean_urban_area_km2": round(float(floodlike_urban.sum()) * px_km2, 3),
    }

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    panels = [
        (axes[0], pre, "Pre VV (dB)", "gray", -22, 0),
        (axes[1], post, "Post VV (dB)", "gray", -22, 0),
        (axes[2], diff, "Post - pre (dB)", "RdBu", -8, 8),
        (axes[3], np.where(floodlike_clean, 1, 0), "Flood-like (cleaned)", "gray", 0, 1),
    ]
    for ax, data, title, cmap, vmin, vmax in panels:
        im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(title)
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle(f"Candidate gate: {spec['label']} ({name})")
    fig.tight_layout()
    fig_path = rel_path(cfg, "outputs") / "figures" / f"candidate_{name}.png"
    fig_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(fig_path, dpi=140)
    plt.close(fig)
    return stats

In [7]:
client = stac_client(cfg)
transform, width, height, dst_crs = target_grid(cfg)
px_km2 = (cfg["grid"]["resolution_m"] ** 2) / 1e6
min_px = cfg["flood_masks"]["morphology"]["min_component_px"]

jrc = rasterio.open(raw / "reference" / "jrc_occurrence_aoi.tif").read(1)
lc = rasterio.open(raw / "landcover" / "worldcover_aoi.tif").read(1)
dem = rasterio.open(raw / "dem" / "dem_aoi.tif").read(1)

rows = []
for name, spec in CANDIDATES.items():
    out_dir = fetch_event(cfg, client, name, spec, dst_crs, transform, width, height, root)
    rows.append(gate(cfg, name, spec, out_dir, jrc, lc, dem, px_km2, min_px))

table = rel_path(cfg, "outputs") / "tables" / "candidate_gate.csv"
table.parent.mkdir(parents=True, exist_ok=True)
with open(table, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

for row in rows:
    print(f"  {row['event']} ({row['label']}): clean flood-like "
          f"{row['floodlike_clean_area_km2']} km^2, urban "
          f"{row['floodlike_clean_urban_area_km2']} km^2")
print("wrote", table)


=== may19: 19-20 May 2025 ===
  pre: S1A_IW_GRDH_1SDV_20250507T115710_20250507T115735_059088_0754A4_rtc
    wrote pre_vv.tif
    wrote pre_vh.tif
  post: S1A_IW_GRDH_1SDV_20250519T115709_20250519T115734_059263_rtc
    wrote post_vv.tif
    wrote post_vh.tif

=== may31: 31 May 2025 ===
  pre: S1A_IW_GRDH_1SDV_20250521T234717_20250521T234742_059299_075C07_rtc
    wrote pre_vv.tif
    wrote pre_vh.tif
  post: S1A_IW_GRDH_1SDV_20250602T234717_20250602T234742_059474_rtc
    wrote post_vv.tif
    wrote post_vh.tif
  may19 (19-20 May 2025): clean flood-like 1.151 km^2, urban 0.0 km^2
  may31 (31 May 2025): clean flood-like 2.357 km^2, urban 0.011 km^2
wrote D:\DIVP PROJECT\outputs\tables\candidate_gate.csv


## Step 3 — Sentinel-2 comparison for the 31 May 2025 candidate

In [8]:
out_dir = Path(cfg["_root"]) / "data" / "candidates" / "may31"
out_dir.mkdir(parents=True, exist_ok=True)

items = list(
    client.search(
        collections=[cfg["stac"]["collections"]["s2_l2a"]],
        bbox=cfg["aoi"]["bbox"],
        datetime=f"2025-06-09/{next_day('2025-06-09')}",
        max_items=10,
    ).items()
)
print("S2 tiles:", [it.id for it in items])
signed = [sign_item(it) for it in items]

for band, resampling in (("B03", Resampling.bilinear), ("B11", Resampling.bilinear), ("SCL", Resampling.nearest)):
    arrays = []
    for item in signed:
        arr = read_reprojected(
            item.assets[band].href, dst_crs, transform, width, height,
            resampling, dst_nodata=0,
        )
        if arr is not None:
            arrays.append(arr)
    mosaic = combine_mosaics(arrays, nodata=0)
    write_geotiff(out_dir / f"s2_{band}.tif", mosaic, transform, dst_crs, 0)
    print(f"  wrote s2_{band}.tif")

pre = to_db(rasterio.open(out_dir / "pre_vv.tif").read(1).astype("float32"))
post = to_db(rasterio.open(out_dir / "post_vv.tif").read(1).astype("float32"))
diff = post - pre
valid = np.isfinite(pre) & np.isfinite(post)

b03 = rasterio.open(out_dir / "s2_B03.tif").read(1).astype("float32")
b11 = rasterio.open(out_dir / "s2_B11.tif").read(1).astype("float32")
scl = rasterio.open(out_dir / "s2_SCL.tif").read(1)
jrc = rasterio.open(rel_path(cfg, "data_raw") / "reference" / "jrc_occurrence_aoi.tif").read(1)

clear = np.isin(scl, [2, 4, 5, 6, 7]) & (b03 > 0) & (b11 > 0)
denom = b03 + b11
mndwi = np.full(b03.shape, np.nan, dtype="float32")
ok = clear & (denom > 0)
mndwi[ok] = (b03[ok] - b11[ok]) / denom[ok]
s2_water = np.isfinite(mndwi) & (mndwi > 0)
permanent = jrc.astype("float32") >= cfg["masks"]["permanent_water"]["occurrence_threshold"]
s2_water_np = s2_water & ~permanent

dark16 = valid & (post < -16.0)
floodlike = dark16 & (diff < -2.0) & ~permanent

def frac(mask, base):
    return float(mask[base].mean()) if base.sum() else float("nan")

stats = {
    "s2_clear_fraction": round(float(clear.mean()), 4),
    "s2_water_fraction_clear": round(frac(s2_water, clear), 4),
    "s2_water_fraction_clear_nonpermanent": round(frac(s2_water_np, clear & ~permanent), 4),
    "sar_dark16_where_s2_water": round(frac(dark16, s2_water), 4),
    "sar_dark16_where_s2_nonwater": round(frac(dark16, clear & ~s2_water), 4),
    "sar_floodlike_where_s2_water": round(frac(floodlike, s2_water), 4),
    "sar_floodlike_where_s2_nonwater": round(frac(floodlike, clear & ~s2_water), 4),
}
for key, value in stats.items():
    print(f"  {key}: {value}")

table = rel_path(cfg, "outputs") / "tables" / "candidate_may31_s2.csv"
with open(table, "w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["metric", "value"])
    for key, value in stats.items():
        writer.writerow([key, value])

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
panels = [
    (axes[0], post, "S1 post VV (dB)", "gray", -22, 0),
    (axes[1], mndwi, "S2 MNDWI (9 Jun 2025)", "RdYlBu", -0.5, 0.5),
    (axes[2], np.where(s2_water_np, 1, 0), "S2 water (non-permanent)", "gray", 0, 1),
    (axes[3], np.where(floodlike, 1, 0), "S1 flood-like", "gray", 0, 1),
]
for ax, data, title, cmap, vmin, vmax in panels:
    im = ax.imshow(data, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.suptitle("31 May 2025 candidate: S1 vs Sentinel-2 (9 Jun 2025)")
fig.tight_layout()
fig_path = rel_path(cfg, "outputs") / "figures" / "candidate_may31_s2.png"
fig.savefig(fig_path, dpi=140)
plt.close(fig)
print("wrote", table)
print("wrote", fig_path)

S2 tiles: ['S2C_MSIL2A_20250609T042721_R133_T46RCQ_20250609T090214', 'S2C_MSIL2A_20250609T042721_R133_T46RCP_20250609T090214']
  wrote s2_B03.tif
  wrote s2_B11.tif
  wrote s2_SCL.tif
  s2_clear_fraction: 0.9376
  s2_water_fraction_clear: 0.1247
  s2_water_fraction_clear_nonpermanent: 0.0378
  sar_dark16_where_s2_water: 0.7793
  sar_dark16_where_s2_nonwater: 0.0121
  sar_floodlike_where_s2_water: 0.0845
  sar_floodlike_where_s2_nonwater: 0.0088
wrote D:\DIVP PROJECT\outputs\tables\candidate_may31_s2.csv
wrote D:\DIVP PROJECT\outputs\figures\candidate_may31_s2.png
